In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df1 = pd.read_pickle(r"/Users/sachuriga/Desktop/Projects/CR_CA1_paper/tables/lfp_with_gamma_event_coupling.pkl")
data = []
for idx, row in df1.iterrows():
    animal_id = row['animal_id']
    s = row['session_id'].split("_")
    if s[3]=="A":
        data.append(row)

df = pd.DataFrame(data)

# --- Add theta/gamma band-power sums to the ORIGINAL df ---
common_frequencies = np.linspace(1, 151, 75) 
# Bands (Hz)
THETA_BAND = (4.0, 12.0)
SLOW_GAMMA_BAND = (20.0, 40)
GAMMA_BAND = (40.0, 91.0)

# LFP columns you already use
lfp_cols = ["lfp_py_norm_run","lfp_sr_norm_run","lfp_py_norm_rest","lfp_sr_norm_rest"]

def sum_band_power_from_cell(cell, band, common_frequencies):
    """
    Extracts the power spectrum from a df cell (expected: [pd.Series]) and returns
    the SUM of power within the specified band after aligning to common_frequencies.
    Returns np.nan when data are missing/ill-formed.
    """
    #try:
    if isinstance(cell, list) and len(cell) > 0 and isinstance(cell[0], pd.Series):
        s = cell[0]
        freqs_src = s.index.values.astype(float)
        power_src = s.values.astype(float)

        # Align to target frequency grid if needed
        if not np.array_equal(freqs_src, common_frequencies):
            power_interp = np.interp(
                common_frequencies, freqs_src, power_src,
                left=np.nan, right=np.nan
            )
            freqs = common_frequencies
            power = power_interp
        else:
            freqs = freqs_src
            power = power_src

        # Band mask (inclusive)
        mask = (freqs >= band[0]) & (freqs <= band[1])
        if not np.any(mask):
            return np.nan

        # Sum of power in band (as requested). If you prefer area, use np.trapz instead.
        band_values = power[mask]
        return np.nansum(band_values)
    # except Exception:
    #     pass
    # return np.nan

# Compute and attach columns
for col in lfp_cols:
    theta_col = f"{col}_theta_sum"
    gamma_col = f"{col}_fast_gamma_sum"
    slow_gamma_col = f"{col}_slow_gamma_sum"
    df[theta_col] = df[col].apply(lambda cell: sum_band_power_from_cell(cell, THETA_BAND, common_frequencies))
    df[gamma_col] = df[col].apply(lambda cell: sum_band_power_from_cell(cell, GAMMA_BAND, common_frequencies))
    df[slow_gamma_col] = df[col].apply(lambda cell: sum_band_power_from_cell(cell, SLOW_GAMMA_BAND, common_frequencies))

# (Optional) quick peek
print(df[[c for col in lfp_cols for c in (f"{col}_theta_sum", f"{col}_fast_gamma_sum")]].head())

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scipy.stats
from scipy.stats import shapiro, ttest_ind, mannwhitneyu
from statsmodels.stats.multitest import multipletests
import ast
from pandas.api.types import is_scalar

# ==========================================
# 1. CONFIGURATION & STYLE
# ==========================================
# GLOBAL FONT SIZE CONTROL
FONT_SIZE = 7

# Standard text arguments to force compliance
TEXT_KWARGS = {'fontsize': FONT_SIZE, 'color': 'black'}

# Update global Matplotlib defaults
plt.rcParams.update({
    'font.size': FONT_SIZE,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Calibri', 'DejaVu Sans', 'sans-serif'],
    'axes.labelsize': FONT_SIZE,
    'axes.titlesize': FONT_SIZE,
    'xtick.labelsize': FONT_SIZE,
    'ytick.labelsize': FONT_SIZE,
    'legend.fontsize': FONT_SIZE,
    'axes.labelpad': 5,
    'ytick.major.pad': 2,
    'xtick.major.pad': 2,
    'axes.edgecolor': 'black',
    'axes.labelcolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'figure.dpi': 1200
})

# IDs
CONTROL_IDS = ['65165', '65091', '63383', '66539', '65622']
EXP_IDS = ['65588', '63385', '66538', '66537', '66922']

# Common frequency grid for interpolation
COMMON_FREQS = np.linspace(1, 151, 75)

# ==========================================
# 2. DATA PROCESSING HELPERS
# ==========================================

def collapse_cell_safe(x, delim=None, strategy="mean"):
    """Parses mixed types in DataFrame cells to a single scalar."""
    if is_scalar(x) or isinstance(x, pd.Timestamp):
        return x

    if isinstance(x, str):
        if delim and delim in x:
            vals = [t.strip() for t in x.split(delim) if t.strip() != ""]
        else:
            if x[:1] in "[({" and x[-1:] in "])}":
                try:
                    x = ast.literal_eval(x)
                except Exception:
                    return x
            else:
                return x

    if isinstance(x, (list, tuple, set, np.ndarray, pd.Series)):
        vals = list(x)
    elif "vals" not in locals():
        return x

    if not vals:
        return np.nan

    nums = []
    for v in vals:
        try:
            nums.append(float(v))
        except Exception:
            pass

    if nums:
        if strategy == "mean": return float(np.mean(nums))
        return float(np.mean(nums)) 

    return vals[0] if strategy != "last" else vals[-1]

def get_lfp_averages(df, animal_ids, column_name):
    """Interpolates and averages LFP power vectors per animal."""
    animal_averages = {}
    
    for animal_id in animal_ids:
        animal_data = df[df['animal_id'] == animal_id][column_name]
        all_power_values = []

        for power_vector in animal_data:
            if isinstance(power_vector, list) and power_vector:
                if isinstance(power_vector[0], pd.Series):
                    power_series = power_vector[0]
                    indices = power_series.index
                    power_values = power_series.values
                    
                    if not np.array_equal(indices, COMMON_FREQS):
                        interp_power = np.interp(COMMON_FREQS, indices, power_values, left=np.nan, right=np.nan)
                    else:
                        interp_power = power_values
                    
                    if len(interp_power) == len(COMMON_FREQS):
                        all_power_values.append(interp_power)

        if all_power_values:
            try:
                stacked = np.vstack(all_power_values)
                animal_averages[animal_id] = np.nanmean(stacked, axis=0)
            except ValueError:
                animal_averages[animal_id] = np.full(len(COMMON_FREQS), np.nan)
        else:
            animal_averages[animal_id] = np.full(len(COMMON_FREQS), np.nan)
            
    return animal_averages

def prepare_scalar_data(df, variables):
    """Extracts and cleans scalar variables for box/violin plots."""
    data = []
    for idx, row in df.iterrows():
        aid = row['animal_id']
        cond = "Control" if aid in CONTROL_IDS else "Exp" if aid in EXP_IDS else None
        
        if cond:
            row_data = {"condition": cond, "animal_id": aid}
            for var in variables:
                row_data[var] = row[var]
            data.append(row_data)

    data_df = pd.DataFrame(data)
    return data_df.map(lambda v: collapse_cell_safe(v, delim=",", strategy="mean"))

# ==========================================
# 3. STATISTICAL HELPERS
# ==========================================

def run_statistical_test(ctrl_data, exp_data):
    """Runs Shapiro to choose between T-test and Mann-Whitney U."""
    stat_c, p_c = shapiro(ctrl_data)
    stat_e, p_e = shapiro(exp_data)
    
    if p_c > 0.05 and p_e > 0.05:
        stat, p_val = ttest_ind(ctrl_data, exp_data)
        test_name = 't-test'
    else:
        stat, p_val = mannwhitneyu(ctrl_data, exp_data)
        test_name = 'Mann-Whitney U'
        
    return p_val, test_name

def calculate_fdr(control_powers, exp_powers):
    """Calculates FDR corrected p-values across frequencies."""
    p_values = []
    num_freqs = control_powers.shape[1]
    
    for idx in range(num_freqs):
        ctrl = control_powers[:, idx]
        ex = exp_powers[:, idx]
        ctrl = ctrl[~np.isnan(ctrl)]
        ex = ex[~np.isnan(ex)]
        
        if len(ctrl) >= 2 and len(ex) >= 2:
            _, p = scipy.stats.ttest_ind(ctrl, ex)
            p_values.append(p)
        else:
            p_values.append(np.nan)

    p_values = np.array(p_values)
    valid_mask = ~np.isnan(p_values)
    q_full = np.full_like(p_values, np.nan)
    
    if np.any(valid_mask):
        reject, q_values, _, _ = multipletests(p_values[valid_mask], method='fdr_bh')
        q_full[valid_mask] = q_values
        
    sig_mask = q_full < 0.05
    return COMMON_FREQS[sig_mask]

# ==========================================
# 4. PLOTTING FUNCTIONS
# ==========================================

def setup_figure():
    """Defines the grid layout and returns figure + dictionary of axes."""
    fig = plt.figure(figsize=(7.2, 10))
    
    outer = gridspec.GridSpec(nrows=5, ncols=4, height_ratios=[1]*5, width_ratios=[1]*4)
    sub = gridspec.GridSpecFromSubplotSpec(
        nrows=1, ncols=3, subplot_spec=outer[3, :], wspace=1, hspace=1, width_ratios=[1]*3
    )

    axes = {}
    axes['spectro_1'] = fig.add_subplot(outer[0, 0:2])
    axes['spectro_2'] = fig.add_subplot(outer[0, 2:4])
    axes['pcm_ctrl'] = fig.add_subplot(outer[1, 0:2])
    axes['pcm_exp']  = fig.add_subplot(outer[1, 2:4])
    axes['comod_ctrl'] = fig.add_subplot(outer[2, 0:2])
    axes['comod_exp']  = fig.add_subplot(outer[2, 2:4])
    
    axes['sub_1'] = fig.add_subplot(sub[0, 0])
    axes['sub_2'] = fig.add_subplot(sub[0, 1])
    axes['sub_3'] = fig.add_subplot(sub[0, 2])
    
    axes['btm_1'] = fig.add_subplot(outer[4, 0])
    axes['btm_2'] = fig.add_subplot(outer[4, 1])
    axes['btm_3'] = fig.add_subplot(outer[4, 2])
    axes['btm_4'] = fig.add_subplot(outer[4, 3])

    axes['lfp_lines'] = [axes['spectro_1'], axes['spectro_2']]
    axes['scalars'] = [
        axes['btm_1'], axes['btm_2'], axes['btm_3'], axes['btm_4'], 
        axes['sub_1'], axes['sub_2'], axes['sub_3']
    ]
    return fig, axes

def plot_lfp_lines(ax, df_lfp, sig_freqs):
    plot_data = []
    for _, row in df_lfp.iterrows():
        if isinstance(row['average_lfp_power'], np.ndarray) and not np.all(np.isnan(row['average_lfp_power'])):
            for f_idx, p in zip(COMMON_FREQS, row['average_lfp_power']):
                plot_data.append({
                    'condition': row['condition'],
                    'frequency': f_idx,
                    'power': p
                })
    
    plot_df = pd.DataFrame(plot_data)

    sns.lineplot(
        data=plot_df, x='frequency', y='power', hue='condition', style='condition',
        palette={'Control': 'blue', 'Experimental': 'red'},
        markers=False, dashes=False, ax=ax, legend=False
    )

    ax.set_xlabel("Frequency (Hz)", **TEXT_KWARGS)
    ax.set_ylabel("Power", **TEXT_KWARGS) # Added label since it's no longer log
    
    # --- MODIFICATIONS START HERE ---
    # ax.set_yscale('log')  <-- Removed
    ax.set_ylim(0, 0.1)    # Set a linear range (adjust 0.05 based on your data peaks)
    # --------------------------------
    
    ax.set_xlim(0, 150)
    ax.grid(False)
    ax.tick_params(labelsize=FONT_SIZE)
    
    if len(sig_freqs) > 0:
        # Adjusted y_pos for linear scale (placing stars near the top)
        y_pos = ax.get_ylim()[1] * 0.9 
        ax.scatter(sig_freqs, [y_pos] * len(sig_freqs), color='green', s=10, marker='*')
        
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

def plot_scalar_comparison(ax, df, var_name, title):
    sns.violinplot(
        data=df, x='condition', y=var_name, ax=ax, inner=None,
        palette={"Control": 'blue', "Exp": 'red'}, width=0.8, cut=0, linewidth=0
    )
    sns.boxplot(
        data=df, x='condition', y=var_name, ax=ax,
        palette={"Control": "black", "Exp": "black"},
        width=0.3, fill=False, showfliers=False, showmeans=False, linewidth=1
    )
    
    ax.set_xlabel('')
    ax.set_ylabel(title, **TEXT_KWARGS)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['CR;DTA-', 'CR;DTA+'], rotation=-45, **TEXT_KWARGS)
    ax.tick_params(labelsize=FONT_SIZE)
    
    c_data = df[df['condition'] == 'Control'][var_name].dropna()
    e_data = df[df['condition'] == 'Exp'][var_name].dropna()
    
    if len(c_data) > 0 and len(e_data) > 0:
        p_val, _ = run_statistical_test(c_data, e_data)
        y_max = ax.get_ylim()[1]
        bar_height = y_max * 0.1
        
        sig_label = ''
        if p_val < 0.001: sig_label = '***'
        elif p_val < 0.01: sig_label = '**'
        elif p_val < 0.05: sig_label = '*'
            
        if sig_label:
            ax.plot([0, 1], [y_max + bar_height, y_max + bar_height], color='black', lw=1.5)
            ax.text(0.5, y_max + bar_height * 1.1, sig_label, ha='center', va='bottom', **TEXT_KWARGS)

# ==========================================
# 5. MAIN EXECUTION
# ==========================================
# NOTE: Ensure df, phases, freqs, norm_spectrogram, etc., are loaded.

fig, axes = setup_figure()

# --- A. Spectrograms (Row 2) ---
try:
    pcm = axes['pcm_ctrl'].pcolormesh(phases, freqs, norm_spectrogram, shading='gouraud', cmap='jet', vmin=0.5, vmax=2)
    cbar1 = plt.colorbar(pcm, ax=axes['pcm_ctrl'])
    cbar1.ax.tick_params(labelsize=FONT_SIZE) # Fix Colorbar font
    axes['pcm_ctrl'].set_xticks([145,145+180])
    axes['pcm_ctrl'].set_xticklabels([0,180],fontsize=FONT_SIZE)
    axes['pcm_ctrl'].set_xlabel('Theta Phase (deg)', **TEXT_KWARGS)
    axes['pcm_ctrl'].set_ylabel('Frequency (Hz)', **TEXT_KWARGS)

    pcm_exp = axes['pcm_exp'].pcolormesh(phases_exp, freqs_exp, norm_spectrogram_exp, shading='gouraud', cmap='jet', vmin=0.5, vmax=2)
    cbar2 = plt.colorbar(pcm_exp, ax=axes['pcm_exp'])
    cbar2.ax.tick_params(labelsize=FONT_SIZE) # Fix Colorbar font
    axes['pcm_exp'].set_xticks([145,145+180])
    axes['pcm_exp'].set_xticklabels([0,180],fontsize=FONT_SIZE)
    axes['pcm_exp'].set_xlabel('Theta Phase (deg)', **TEXT_KWARGS)
    axes['pcm_exp'].set_ylabel('Frequency (Hz)', **TEXT_KWARGS)

except NameError:
    print("Skipping Spectrograms (variables missing)")

# --- B. Comodulograms (Row 3) - FIXED FONT SIZE ---
# --- B. Comodulograms (Row 3) - FIXED COLORBAR FONT SIZE ---
try:
    # 1. Control Animal
    # Set current axis to the correct subplot
    ax_ctrl = axes['comod_ctrl']
    plt.sca(ax_ctrl)
    
    # Plot with colorbar=False (we will add it manually)
    p_control.comodulogram(xpac.mean(axis=-1), cmap='jet', plotas='imshow', 
                           title='', vmin=0, vmax=0.35, colorbar=False)
    
    # Manually add colorbar to control font size
    mappable_ctrl = ax_ctrl.images[0] # Grab the image data from the plot
    cbar_c = plt.colorbar(mappable_ctrl, ax=ax_ctrl)
    cbar_c.ax.tick_params(labelsize=FONT_SIZE) # <--- THIS FIXES THE FONT SIZE
    
    # Labels
    #ax_ctrl.set_title('Phase-Power Comodulogram (CA1)', **TEXT_KWARGS)
    ax_ctrl.set_xlabel('Phase Frequency (Hz)', **TEXT_KWARGS)
    ax_ctrl.set_ylabel('Amplitude Frequency (Hz)', **TEXT_KWARGS)
    ax_ctrl.tick_params(axis='both', which='major', labelsize=FONT_SIZE)
    
    # 2. Experimental Animal
    # Set current axis
    ax_exp = axes['comod_exp']
    plt.sca(ax_exp)
    
    # Plot with colorbar=False
    p_exp.comodulogram(xpac_exp.mean(axis=-1), cmap='jet', plotas='imshow', 
                       title='', vmin=0, vmax=0.35, colorbar=False)
    
    # Manually add colorbar
    mappable_exp = ax_exp.images[0]
    cbar_e = plt.colorbar(mappable_exp, ax=ax_exp)
    cbar_e.ax.tick_params(labelsize=FONT_SIZE) # <--- THIS FIXES THE FONT SIZE
    
    # Labels
    #ax_exp.set_title('Phase-Power Comodulogram (CA1)', **TEXT_KWARGS)
    ax_exp.set_xlabel('Phase Frequency (Hz)', **TEXT_KWARGS)
    ax_exp.set_ylabel('Amplitude Frequency (Hz)', **TEXT_KWARGS)
    ax_exp.tick_params(axis='both', which='major', labelsize=FONT_SIZE)

except NameError:
    print("Skipping Comodulograms (variables missing)")
except Exception as e:
    print(f"Error plotting comodulogram: {e}")

# --- C. LFP Line Plots (Row 1) ---
unique_animals = np.unique(df['animal_id'])
lfp_types = ["lfp_py_norm_run", "lfp_py_norm_rest"]

for i, lfp_col in enumerate(lfp_types):
    if i < len(axes['lfp_lines']):
        ax = axes['lfp_lines'][i]
        avg_dict = get_lfp_averages(df, unique_animals, lfp_col)
        lfp_df = pd.DataFrame({"animal_id": avg_dict.keys(), "average_lfp_power": avg_dict.values()})
        lfp_df['condition'] = lfp_df['animal_id'].apply(lambda x: 'Control' if x in CONTROL_IDS else 'Experimental' if x in EXP_IDS else 'Unknown')
        lfp_df = lfp_df[lfp_df['condition'] != 'Unknown']
        
        ctrl_powers = np.stack(lfp_df[lfp_df['condition'] == 'Control']['average_lfp_power'].values)
        exp_powers = np.stack(lfp_df[lfp_df['condition'] == 'Experimental']['average_lfp_power'].values)
        plot_lfp_lines(ax, lfp_df, calculate_fdr(ctrl_powers, exp_powers))

# --- D. Scalar Variables (Rows 4 & 5) ---
scalar_vars = [
    'slow_event_rate_py', 'slow_theta_gamma_coupling_py', 'fast_event_rate_py', 
    'fast_theta_gamma_coupling_py', "lfp_py_norm_run_theta_sum", 
    "lfp_py_norm_run_slow_gamma_sum", "lfp_py_norm_run_fast_gamma_sum"
]
scalar_titles = [
    'Episodes/s', 'Vector length', 'Episodes/s', 'Vector length', 
    'Normalized power', 'Normalized power', 'Normalized power'
]

unpacked_df = prepare_scalar_data(df, scalar_vars)

for i, var in enumerate(scalar_vars):
    if i < len(axes['scalars']):
        plot_scalar_comparison(axes['scalars'][i], unpacked_df, var, scalar_titles[i])

plt.tight_layout()
save_path = r'/Users/sachuriga/Desktop/Projects/CR_CA1_paper/Figures_mac/suppfig10.png'
plt.savefig(save_path, transparent=True, dpi=1200, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import ast
from pandas.api.types import is_scalar

def collapse_cell_safe(x, delim=None, strategy="mean"):
    # 1) Scalars: return as-is (including NaT/Timestamps)
    if is_scalar(x) or isinstance(x, pd.Timestamp):
        return x

    # 2) Strings: handle delimited or stringified lists
    if isinstance(x, str):
        if delim and delim in x:
            vals = [t.strip() for t in x.split(delim) if t.strip() != ""]
        else:
            # try to parse "[1, 2]" / "(1,2)" / "{1,2}" safely
            if x[:1] in "[({" and x[-1:] in "])}":
                try:
                    x = ast.literal_eval(x)
                except Exception:
                    return x
            else:
                return x
    # 3) Iterables: lists/tuples/sets/arrays/Series
    if isinstance(x, (list, tuple, set, np.ndarray, pd.Series)):
        vals = list(x)
    elif "vals" not in locals():
        # unknown type, leave it
        return x

    if not vals:
        return np.nan

    # Try numeric aggregation; ignore non-numeric entries
    nums = []
    for v in vals:
        try:
            nums.append(float(v))
        except Exception:
            pass

    if nums:
        if strategy == "mean":   return float(np.mean(nums))
        if strategy == "median": return float(np.median(nums))
        if strategy == "min":    return float(np.min(nums))
        if strategy == "max":    return float(np.max(nums))
        if strategy == "first":  return vals[0]
        if strategy == "last":   return vals[-1]
        return float(np.mean(nums))  # default

    # If nothing numeric, fall back to first/last
    return vals[0] if strategy != "last" else vals[-1]


# Example: comma-separated numbers -> mean
df_single = data_df.map(lambda v: collapse_cell_safe(v, delim=",", strategy="mean"))



In [ ]:
cols = [
    'slow_event_rate_py', 'slow_theta_gamma_coupling_py',
    'fast_event_rate_py', 'fast_theta_gamma_coupling_py',
    'slow_event_rate_sr', 'slow_theta_gamma_coupling_sr',
    'fast_event_rate_sr', 'fast_theta_gamma_coupling_sr'
]

# (optional) ensure these columns are numeric
df[cols] = df[cols].apply(pd.to_numeric, errors='coerce')

# group by animal id and take the mean
df_avg = (
    df.groupby('animal_id', as_index=False)[cols]
      .mean()
)
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids     = ['65588', '63385', '66538', '66537', '66922']

# ensure ids are strings
df_avg['animal_id'] = df_avg['animal_id'].astype(str)

# option A: with a mapping (cleanest)
mapping = {**{i: 'Control' for i in control_ids},
           **{i: 'Exp'     for i in exp_ids}}
df_avg['condition'] = df_avg['animal_id'].map(mapping).fillna('Unknown')

# If you already computed df_avg (from your previous step), add the same column:
df_avg['condition'] = df_avg['animal_id'].map(mapping).fillna('Unknown')

In [ ]:
import numpy as np
from scipy.signal import butter, sosfiltfilt, argrelextrema, hilbert
import matplotlib.pyplot as plt

def bandpass_filter(data, lowcut, highcut, fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    sos = butter(order, [low, high], analog=False, btype='band', output='sos')
    y = sosfiltfilt(sos, data)
    return y

import numpy as np
from scipy.signal import hilbert, argrelextrema

# Assuming helper function 'bandpass_filter' is defined elsewhere in your code
# from your_utils import bandpass_filter 

def generate_normalized_spectrogram(signal, fs=1250, freq_step=2, num_phase_bins=12, start_phase=0, peak_at_deg=90):
    """
    Generates a phase-amplitude spectrogram with smooth cyclic connections.
    
    Args:
        peak_at_deg (int): The angle (in degrees) where the Peak of the theta cycle should be located.
                           Default is 135.
    """
    # 1. Bandpass filter for theta (6-12 Hz)
    theta_filt = bandpass_filter(signal, 6, 12, fs)
    
    # 2. Find local minima (troughs) to identify cycles
    minima_idx = argrelextrema(theta_filt, np.less)[0]
    
    cycle_pairs = []
    for i in range(len(minima_idx) - 1):
        start = minima_idx[i]
        end = minima_idx[i + 1]
        dur_ms = (end - start) / fs * 1000
        if 100 <= dur_ms <= 150:
            cycle_pairs.append((start, end))
            
    if not cycle_pairs:
        raise ValueError("No valid theta cycles found.")
    
    # 3. Compute Phase 
    analytic = hilbert(theta_filt)
    phase = np.angle(analytic)
    
    # --- PHASE ALIGNMENT LOGIC ---
    # First, calculate the average phase at the physical troughs
    trough_phases = phase[minima_idx]
    mean_trough_phase = np.angle(np.mean(np.exp(1j * trough_phases)))
    
    # Calculate offset required to move Peak to 135 degrees.
    # Logic: 
    # If we only subtracted mean_trough_phase, Trough would be 0 deg and Peak would be 180 deg.
    # We want Peak to be at 'peak_at_deg' (e.g., 135).
    # Shift = Target - Current = 135 - 180 = -45 degrees.
    target_peak_deg = peak_at_deg
    current_peak_deg = 180 # Based on Trough being 0
    offset_deg = target_peak_deg - current_peak_deg
    offset_rad = np.deg2rad(offset_deg)
    
    # Apply alignment: Normalize Trough to 0, then apply the specific peak shift
    aligned_phase = (phase - mean_trough_phase + offset_rad) % (2 * np.pi)
    # -----------------------------
    
    # 4. Setup Frequencies and Bins
    freqs = np.arange(20, 121, freq_step)
    
    # Standard bins 0 to 2pi
    bin_edges = np.linspace(0, 2 * np.pi, num_phase_bins + 1)
    
    # Accumulators
    power_sum = np.zeros((len(freqs), num_phase_bins))
    count = np.zeros((len(freqs), num_phase_bins))
    
    dt = 1 / fs
    
    # 5. Collect Data
    for start, end in cycle_pairs:
        segment = signal[start:end]
        cycle_phases = aligned_phase[start:end]
        
        for f_idx, f in enumerate(freqs):
            # Morlet Wavelet parameters
            sigma_f = f / 7.0
            sigma_t = 1 / (2 * np.pi * sigma_f)
            t_wave = np.arange(-4 * sigma_t, 4 * sigma_t + dt, dt)
            A = 1 / np.sqrt(sigma_t * np.sqrt(np.pi))
            wavelet = A * np.exp(-t_wave**2 / (2 * sigma_t**2)) * np.exp(2j * np.pi * f * t_wave)
            
            # Convolve
            pad_left = (len(wavelet) - 1) // 2
            pad_right = len(wavelet) - 1 - pad_left
            padded_segment = np.pad(segment, (pad_left, pad_right), mode='reflect')
            conv = np.convolve(padded_segment, wavelet, mode='valid')
            power = np.abs(conv)**2
            
            # Binning
            bin_idx = np.digitize(cycle_phases, bin_edges) - 1
            
            # Vectorized bin accumulation for speed
            for b in range(num_phase_bins):
                mask = (bin_idx == b)
                if np.any(mask):
                    s = np.sum(power[mask])
                    c = np.sum(mask)
                    power_sum[f_idx, b] += s
                    count[f_idx, b] += c

    # 6. Normalize
    avg_power = np.divide(power_sum, count, where=count > 0)
    avg_power[count == 0] = np.nan
    
    norm_spectrogram = np.zeros_like(avg_power)
    for f_idx in range(len(freqs)):
        mean_p = np.nanmean(avg_power[f_idx, :])
        if mean_p > 0:
            norm_spectrogram[f_idx, :] = avg_power[f_idx, :] / mean_p
        else:
            norm_spectrogram[f_idx, :] = np.nan

    # ============================================================
    # Visualization Adjustment (Rolling)
    # ============================================================
    
    # Note: 'start_phase' dictates the X-axis of the plot (viewing window).
    # 'peak_at_deg' dictates where the data lands within that window.
    
    deg_per_bin = 360 / num_phase_bins
    shift_bins = int(start_phase / deg_per_bin)
    
    # Roll the data
    shifted_spectrogram = np.roll(norm_spectrogram, -shift_bins, axis=1)
    
    # Pad for continuity
    plot_data = np.hstack((shifted_spectrogram, shifted_spectrogram[:, :1]))
    
    # Create Plotting Edges
    plot_phase_edges = np.linspace(start_phase, start_phase + 360, num_phase_bins + 1)
    
    return freqs, plot_phase_edges, plot_data
# Example usage (replace with your actual signal)
# signal = your_lfp_data_here  # np.array of LFP signal

# For demonstration, generate a sample signal (replace with real data)
fs = 1250
eeg = df1['lfp_py'][6][0].values
signal = eeg
freqs, phases, norm_spectrogram = generate_normalized_spectrogram(signal, fs)
# Plot the figure
fig, ax = plt.subplots(figsize=(8, 6))
pcm = ax.pcolormesh(phases, freqs, norm_spectrogram, shading='gouraud', cmap='jet',vmin=0.5,vmax=2)
cbar = plt.colorbar(pcm, ax=ax)
cbar.set_label('Normalized Power')
ax.set_xlabel('Theta Phase (degrees)')
ax.set_ylabel('Frequency (Hz)')
ax.set_title('Normalized Power Spectrogram Averaged Across Theta Cycles')
plt.show()

In [ ]:
df1['animal_id'][6]

In [ ]:
import os 
os.chdir(r'/Users/sachuriga/Desktop/code/nwb4fp/SRC')

from nwb4fp.analyses.data import pos2speed,speed_filtered_spikes,load_speed_fromNWB,load_units_fromNWB,find_run_indices
import pynapple as nap
i=3
smoothed_speed = df1['smoothed_speed'][i]
time_stemp = df1['time_stemp'][i]
starts,stops = find_run_indices(smoothed_speed, threshold=0.05)
run_ep = nap.IntervalSet(start=time_stemp[starts], end=time_stemp[stops], time_units='s')
eeg = df1['lfp_py'][i][0].values

In [ ]:
# Plot the figure
fig, ax = plt.subplots(figsize=(8, 6))
signal_exp = df1['lfp_py'][85][1].values
freqs_exp, phases_exp, norm_spectrogram_exp = generate_normalized_spectrogram(signal_exp, fs)
pcm_exp = ax.pcolormesh(phases_exp, freqs_exp, norm_spectrogram_exp, shading='gouraud', cmap='jet',vmin=0.5,vmax=2)
cbar = plt.colorbar(pcm_exp, ax=ax)
cbar.set_label('Normalized Power')
ax.set_xlabel('Theta Phase (degrees)')
ax.set_ylabel('Frequency (Hz)')
ax.set_title('Normalized Power Spectrogram Averaged Across Theta Cycles')
plt.show()

In [ ]:
df1['lfp_py'][85]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, hilbert

def get_phase_amplitude_coupling(lfp_data, fs):
    """
    Generates a Comodulogram (Phase-Amplitude Coupling Heatmap).
    """
    
    # --- 1. Define Frequency Ranges ---
    # Theta (Phase) frequencies: 4 to 12 Hz (step 1 Hz)
    phase_freqs = np.arange(4, 13, 1)
    
    # Gamma (Amplitude) frequencies: 30 to 180 Hz (step 5 Hz)
    amp_freqs = np.arange(30, 181, 5)
    
    # Matrix to store Modulation Index (MI)
    comodulogram = np.zeros((len(amp_freqs), len(phase_freqs)))

    print("Computing Comodulogram... (This may take a moment)")

    # --- 2. Iterate through Phase Frequencies ---
    for i, p_freq in enumerate(phase_freqs):
        # Filter for Phase (Theta)
        b_p, a_p = butter(3, [p_freq-0.5, p_freq+0.5], btype='bandpass', fs=fs)
        phase_signal = filtfilt(b_p, a_p, lfp_data)
        
        # Extract Phase using Hilbert Transform
        phase_ts = np.angle(hilbert(phase_signal))
        
        # --- 3. Iterate through Amplitude Frequencies ---
        for j, a_freq in enumerate(amp_freqs):
            # Filter for Amplitude (Gamma)
            # Using a wider band for amplitude (e.g., +/- 5Hz or 10Hz) captures the sidebands
            b_a, a_a = butter(3, [a_freq-5, a_freq+5], btype='bandpass', fs=fs)
            amp_signal = filtfilt(b_a, a_a, lfp_data)
            
            # Extract Amplitude envelope
            amp_ts = np.abs(hilbert(amp_signal))
            
            # --- 4. Calculate Modulation Index (Mean Vector Length method) ---
            # Converting to complex vector: Amplitude * exp(i * Phase)
            z = amp_ts * np.exp(1j * phase_ts)
            mvl = np.abs(np.mean(z)) # Mean Vector Length
            
            # Normalize (optional, helps with visualization)
            # Real MI (Tort method) requires binning, but MVL is faster for a quick look
            comodulogram[j, i] = mvl

    # --- 5. Plotting ---
    plt.figure(figsize=(6, 5))
    extent = [phase_freqs[0], phase_freqs[-1], amp_freqs[0], amp_freqs[-1]]
    
    # Plot heatmap with origin at lower left
    plt.imshow(comodulogram, aspect='auto', origin='lower', extent=extent, cmap='jet')
    
    plt.xlabel('Phase Frequency (Hz) [Theta]')
    plt.ylabel('Amplitude Frequency (Hz) [Gamma]')
    plt.title('Theta-Gamma Comodulogram')
    plt.colorbar(label='Coupling Strength')
    plt.show()

# --- Example Usage ---
# Generate a synthetic signal if you don't have a file loaded
fs = 1000  # Sampling freq
t = np.arange(0, 10, 1/fs)
# Construct signal: Theta (8Hz) modulates Gamma (80Hz)
theta = np.sin(2 * np.pi * 8 * t)
gamma = np.sin(2 * np.pi * 80 * t) * ((theta + 1) / 2) # Amplitude modulation
noise = np.random.normal(0, 0.5, len(t))
lfp_synthetic = theta + gamma + noise

# Run the function
get_phase_amplitude_coupling(lfp_synthetic, fs)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorpac import Pac

# --- 1. SETUP YOUR DATA HERE ---
# Replace 'lfp_signal' with your actual data array
# Example: lfp_signal = np.load('my_mPFC_recording.npy')
# If you don't have data loaded, uncomment the line below to simulate a signal for testing
# from tensorpac.signals import pac_signals_tort
# lfp_signal, time = pac_signals_tort(f_pha=8, f_amp=80, n_seconds=10, noise=2, sf=fs)

# --- 2. DEFINE FREQUENCY RANGES ---
# Low frequency (Phase) - typically Theta (4-12 Hz) or broad low freq
phase_freqs = (1, 20, 1, 0.2)  # (start, stop, width, step)

# Fast frequency (Power/Amplitude) - typically Gamma (30-100+ Hz)
amp_freqs = (12, 150, 10, 2) # (start, stop, width, step)

# --- 3. COMPUTE THE COMODULOGRAM ---
# We use idpac=(5, 0, 0) for the standard Modulation Index (Tort et al., 2010)
# This measures how much the high-freq amplitude deviates from a uniform distribution 
# across low-freq phase bins.
p_control = Pac(idpac=(5, 0, 0), f_pha=phase_freqs, f_amp=amp_freqs)

# Filter the data and extract phases/amplitudes
# Note: 'xpac' will contain the 2D comodulogram matrix
xpac = p_control.filterfit(fs, signal)

# --- 4. PLOT ---
plt.figure(figsize=(10, 8))
p_control.comodulogram(xpac.mean(axis=-1), cmap='jet', plotas='imshow', title='Phase-Power Comodulogram (CA1)',vmin=0,vmax=0.35)
plt.xlabel('Phase Frequency (Hz) [Low Freq]')
plt.ylabel('Amplitude Frequency (Hz) [Fast Freq]')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorpac import Pac

# --- 1. SETUP YOUR DATA HERE ---
# Replace 'lfp_signal' with your actual data array
# Example: lfp_signal = np.load('my_mPFC_recording.npy')
# If you don't have data loaded, uncomment the line below to simulate a signal for testing
# from tensorpac.signals import pac_signals_tort
# lfp_signal, time = pac_signals_tort(f_pha=8, f_amp=80, n_seconds=10, noise=2, sf=fs)

# --- 2. DEFINE FREQUENCY RANGES ---
# Low frequency (Phase) - typically Theta (4-12 Hz) or broad low freq
phase_freqs = (1, 20, 1, 0.2)  # (start, stop, width, step)

# Fast frequency (Power/Amplitude) - typically Gamma (30-100+ Hz)
amp_freqs = (12, 150, 10, 1) # (start, stop, width, step)

# --- 3. COMPUTE THE COMODULOGRAM ---
# We use idpac=(5, 0, 0) for the standard Modulation Index (Tort et al., 2010)
# This measures how much the high-freq amplitude deviates from a uniform distribution 
# across low-freq phase bins.
p_exp = Pac(idpac=(5, 0, 0), f_pha=phase_freqs, f_amp=amp_freqs)

# Filter the data and extract phases/amplitudes
# Note: 'xpac' will contain the 2D comodulogram matrix
xpac_exp = p_exp.filterfit(fs, signal_exp)

# --- 4. PLOT ---
plt.figure(figsize=(10, 8))
p_exp.comodulogram(xpac_exp.mean(axis=-1), cmap='jet', plotas='imshow', title='Phase-Power Comodulogram (CA1)',vmax=0.35)
plt.xlabel('Phase Frequency (Hz) [Low Freq]')
plt.ylabel('Amplitude Frequency (Hz) [Fast Freq]')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert, butter, filtfilt
from tensorpac.signals import pac_signals_tort # Importing to simulate signal

def compute_aac(lfp, fs, f_start=1, f_stop=100, n_freqs=50, width=2):
    """
    Computes Amplitude-Amplitude Coupling (AAC) matrix with the diagonal masked.
    """
    # 1. Define frequency range
    freqs = np.linspace(f_start, f_stop, n_freqs)
    
    # 2. Pre-allocate envelope matrix (n_freqs x n_samples)
    envelopes = np.zeros((n_freqs, len(lfp)))
    
    # 3. Filter bank & Hilbert transform
    print(f"Filtering {n_freqs} frequency bands...")
    for i, f_center in enumerate(freqs):
        low = f_center - width/2
        high = f_center + width/2
        if low <= 0: low = 0.1
        
        b, a = butter(4, [low, high], fs=fs, btype='bandpass')
        filtered = filtfilt(b, a, lfp)
        envelopes[i, :] = np.abs(hilbert(filtered))
    
    # 4. Compute Correlation Matrix
    aac_matrix = np.corrcoef(envelopes)
    
    # --- STEP TO REMOVE DIAGONAL ---
    # We set the diagonal to NaN (Not a Number) so it appears empty/white
    # instead of bright red (correlation=1.0)
    np.fill_diagonal(aac_matrix, np.nan)
    
    return aac_matrix, freqs

# --- USER INPUTS ---

# Simulating data (Theta-Gamma coupling)
#signal, time = pac_signals_tort(f_pha=8, f_amp=80, n_seconds=20, noise=0.5, sf=fs)

# Compute AAC
aac_matrix, freqs = compute_aac(signal, fs, f_start=2, f_stop=120, n_freqs=60, width=4)

# --- PLOT ---
plt.figure(figsize=(9, 8), dpi=300)

# Plot the matrix (NaN values will be transparent/white)
plt.imshow(aac_matrix, extent=[freqs[0], freqs[-1], freqs[0], freqs[-1]], 
           origin='lower', cmap='RdBu_r', vmin=-0.25, vmax=0.25)

plt.colorbar(label='Correlation (Pearson r)')
plt.title('Amplitude-Amplitude Coupling (CA1) - Diag Removed', fontsize=14)
plt.xlabel('Frequency 1 (Hz)')
plt.ylabel('Frequency 2 (Hz)')

plt.tight_layout()
plt.show()

In [ ]:
"""
Phase–Amplitude Coupling (PAC) comodulogram with Tort Modulation Index (MI)

This module provides:
  1) A robust MI implementation (Tort et al., 2010 style) for limited-time datasets
  2) Time-stepped / event-epoch PAC computation
  3) Publication-ready plotting helpers for comodulograms

Primary entry point: `compute_pac_comodulogram` + `plot_comodulogram`.

Author: ChatGPT
"""
from __future__ import annotations
from typing import Iterable, List, Optional, Sequence, Tuple
import numpy as np
from scipy.signal import butter, filtfilt, hilbert
import matplotlib.pyplot as plt

ArrayLike = np.ndarray

# ------------------------------
# Filtering utilities
# ------------------------------
def _butter_bandpass(low: float, high: float, fs: float, order: int = 4) -> Tuple[np.ndarray, np.ndarray]:
    nyq = 0.5 * fs
    low_n = low / nyq
    high_n = high / nyq
    if low_n <= 0 or high_n >= 1 or low_n >= high_n:
        raise ValueError(f"Invalid band [{low}, {high}] Hz for fs={fs} Hz.")
    b, a = butter(order, [low_n, high_n], btype="bandpass")
    return b, a


def bandpass_filter(x: ArrayLike, fs: float, band: Tuple[float, float], order: int = 4) -> ArrayLike:
    """Zero-phase IIR bandpass using filtfilt.

    Parameters
    ----------
    x : array, shape (n_samples,) or (n_channels, n_samples)
    fs : float
        Sampling rate in Hz.
    band : (low, high)
        Frequency band in Hz.
    order : int
        Butterworth order.
    """
    x = np.asarray(x)
    b, a = _butter_bandpass(band[0], band[1], fs, order)
    if x.ndim == 1:
        return filtfilt(b, a, x)
    elif x.ndim == 2:
        return np.vstack([filtfilt(b, a, xi) for xi in x])
    else:
        raise ValueError("x must be 1D or 2D")


# ------------------------------
# Modulation Index (Tort et al.)
# ------------------------------

def modulation_index_tort(phase_series: ArrayLike, amp_envelope: ArrayLike, n_bins: int = 18, eps: float = 1e-10) -> float:
    """Compute Tort modulation index between a phase time series and an amplitude envelope.

    Parameters
    ----------
    phase_series : array
        Instantaneous phase in radians (e.g., angle(hilbert(theta_band))).
    amp_envelope : array
        Instantaneous amplitude envelope (e.g., abs(hilbert(gamma_band))).
    n_bins : int
        Number of phase bins spanning [-pi, pi).
    eps : float
        Small value to avoid log(0).

    Returns
    -------
    MI : float
        Normalized KL divergence between the phase-binned amplitude distribution and uniform.
    """
    phase = np.asarray(phase_series)
    amp = np.asarray(amp_envelope)
    mask = np.isfinite(phase) & np.isfinite(amp)
    phase = phase[mask]
    amp = amp[mask]
    if phase.size < n_bins * 5:
        return np.nan  # too few samples for a stable estimate

    # Bin by phase
    edges = np.linspace(-np.pi, np.pi, n_bins + 1)
    # Map phase to bin indices [0, n_bins-1]
    idx = np.digitize(phase, edges) - 1
    idx[idx == n_bins] = n_bins - 1

    # Mean amplitude per phase bin
    mean_amp = np.zeros(n_bins)
    counts = np.zeros(n_bins, dtype=int)
    for k in range(n_bins):
        sel = idx == k
        counts[k] = sel.sum()
        if counts[k] > 0:
            mean_amp[k] = amp[sel].mean()
        else:
            mean_amp[k] = 0.0

    if counts.sum() == 0:
        return np.nan

    P = mean_amp / (mean_amp.sum() + eps)
    U = np.full(n_bins, 1.0 / n_bins)

    # KL divergence and normalization
    kl = np.sum(P * (np.log(P + eps) - np.log(U + eps)))
    mi = kl / np.log(n_bins)
    return float(mi)


# ------------------------------
# Epoch helpers
# ------------------------------

def make_sliding_epochs(n_samples: int, fs: float, window_s: float, step_s: Optional[float] = None) -> List[Tuple[int, int]]:
    """Create [start, stop) sample windows covering the signal.

    If step_s is None, defaults to window_s/2.
    """
    if step_s is None:
        step_s = window_s / 2
    w = int(round(window_s * fs))
    s = int(round(step_s * fs))
    if w <= 1:
        raise ValueError("window too small")
    starts = np.arange(0, n_samples - w + 1, s)
    return [(int(a), int(a + w)) for a in starts]


# ------------------------------
# Core PAC computation
# ------------------------------

def compute_pac_comodulogram(
    lfp: ArrayLike,
    fs: float,
    phase_bands: Sequence[Tuple[float, float]],
    amp_bands: Sequence[Tuple[float, float]],
    epochs: Optional[Sequence[Tuple[int, int]]] = None,
    window_s: Optional[float] = None,
    step_s: Optional[float] = None,
    n_bins: int = 18,
    filter_order: int = 4,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Optional[np.ndarray]]:
    """Compute PAC MI comodulogram over frequency pairs and (optionally) time epochs.

    You can provide explicit `epochs` as a list of (start, stop) sample indices.
    Alternatively, set `window_s` (and optional `step_s`) to tile the signal.

    Parameters
    ----------
    lfp : array, shape (n_samples,)
        LFP / field potential time series.
    fs : float
        Sampling rate in Hz.
    phase_bands : list of (f_lo, f_hi)
        Bands that provide phase (typically low frequencies, e.g., theta 4–12 Hz).
    amp_bands : list of (f_lo, f_hi)
        Bands whose amplitude is modulated (typically higher, e.g., gamma 30–200 Hz).
    epochs : list of (start, stop) in samples, optional
        Event-aligned epochs. If None and window_s given, sliding windows are used.
    window_s, step_s : float, optional
        Sliding window parameters in seconds.
    n_bins : int
        Number of phase bins for MI.
    filter_order : int
        Butterworth order for bandpass.

    Returns
    -------
    mi : array, shape (n_phase, n_amp, n_epochs)
    phase_centers : array, shape (n_phase,)
    amp_centers : array, shape (n_amp,)
    epoch_times_s : array, shape (n_epochs,), optional
        Epoch center times in seconds (None if `epochs` provided explicitly without timing context).
    """
    x = np.asarray(lfp).astype(float)
    if x.ndim != 1:
        raise ValueError("lfp must be 1D")

    # Epochs
    if epochs is None:
        if window_s is None:
            raise ValueError("Provide `epochs` or `window_s`.")
        epochs = make_sliding_epochs(len(x), fs, window_s, step_s)
        epoch_times_s = np.array([(a + b) / 2 / fs for a, b in epochs])
    else:
        epoch_times_s = np.array([(a + b) / 2 / fs for a, b in epochs])

    nP = len(phase_bands)
    nA = len(amp_bands)
    nE = len(epochs)

    # Pre-filter once per band to avoid re-filtering per epoch
    phase_filt = []  # list of instantaneous phase arrays
    for (flo, fhi) in phase_bands:
        xf = bandpass_filter(x, fs, (flo, fhi), order=filter_order)
        phase_filt.append(np.angle(hilbert(xf)))
    phase_filt = np.stack(phase_filt, axis=0)  # (nP, n_samples)

    amp_filt = []  # list of amplitude envelopes per amp band
    for (flo, fhi) in amp_bands:
        xf = bandpass_filter(x, fs, (flo, fhi), order=filter_order)
        amp_filt.append(np.abs(hilbert(xf)))
    amp_filt = np.stack(amp_filt, axis=0)  # (nA, n_samples)

    # Centers for axes labels
    phase_centers = np.array([(flo + fhi) / 2 for (flo, fhi) in phase_bands], dtype=float)
    amp_centers = np.array([(flo + fhi) / 2 for (flo, fhi) in amp_bands], dtype=float)

    # Compute MI per (phase, amp, epoch)
    mi = np.full((nP, nA, nE), np.nan, dtype=float)
    for e, (a, b) in enumerate(epochs):
        # Slightly shrink window to reduce edge artifacts from Hilbert/filter transients
        pad = int(0.05 * (b - a))  # 5% padding removal
        aa = min(a + pad, b)
        bb = max(a, b - pad)
        if bb - aa < 10:
            continue
        for ip in range(nP):
            ph = phase_filt[ip, aa:bb]
            for ia in range(nA):
                am = amp_filt[ia, aa:bb]
                mi[ip, ia, e] = modulation_index_tort(ph, am, n_bins=n_bins)

    return mi, phase_centers, amp_centers, epoch_times_s


# ------------------------------
# Plotting helpers
# ------------------------------

def plot_comodulogram(
    mi: np.ndarray,
    phase_centers: np.ndarray,
    amp_centers: np.ndarray,
    epoch: Optional[int] = None,
    agg: str = "mean",
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    ax: Optional[plt.Axes] = None,
    cbar: bool = True,
    title: Optional[str] = None,
):
    """Plot a standard comodulogram: phase-freq (x) vs amp-freq (y).

    Parameters
    ----------
    mi : array, shape (nP, nA, nE)
    epoch : int or None
        If None, aggregate across epochs using `agg` ("mean" or "max").
    """
    if mi.ndim != 3:
        raise ValueError("mi must be (nP, nA, nE)")

    if epoch is None:
        if agg == "mean":
            M = np.nanmean(mi, axis=2)
        elif agg == "max":
            M = np.nanmax(mi, axis=2)
        else:
            raise ValueError("agg must be 'mean' or 'max'")
    else:
        if not (0 <= epoch < mi.shape[2]):
            raise IndexError("epoch out of range")
        M = mi[:, :, epoch]

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5), dpi=120)
    im = ax.imshow(
        M.T,
        origin="lower",
        aspect="auto",
        extent=[phase_centers.min(), phase_centers.max(), amp_centers.min(), amp_centers.max()],
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest",
    )
    ax.set_xlabel("Phase frequency (Hz)")
    ax.set_ylabel("Amplitude frequency (Hz)")
    if title:
        ax.set_title(title)
    if cbar:
        plt.colorbar(im, ax=ax, label="Modulation Index (MI)")
    return ax


def plot_time_resolved(
    mi: np.ndarray,
    phase_centers: np.ndarray,
    amp_centers: np.ndarray,
    epoch_times_s: Optional[np.ndarray] = None,
    phase_idx: int = 0,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    ax: Optional[plt.Axes] = None,
    cbar: bool = True,
    title: Optional[str] = None,
):
    """Plot time-resolved PAC for a single phase frequency across amplitude bands.

    Shows amplitude-frequency vs time (epochs on x). Useful for stepping through task time.
    """
    if mi.ndim != 3:
        raise ValueError("mi must be (nP, nA, nE)")
    if not (0 <= phase_idx < mi.shape[0]):
        raise IndexError("phase_idx out of range")

    M = mi[phase_idx, :, :]
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 4), dpi=120)

    extent = [0, M.shape[1], amp_centers.min(), amp_centers.max()]
    if epoch_times_s is not None and len(epoch_times_s) == M.shape[1]:
        # Use epoch indices but we will relabel ticks with times
        x = np.arange(M.shape[1])
        im = ax.imshow(M, origin="lower", aspect="auto", extent=[x.min(), x.max(), amp_centers.min(), amp_centers.max()], vmin=vmin, vmax=vmax, interpolation="nearest")
        # Relabel x ticks
        ticks = np.linspace(x.min(), x.max(), num=min(8, len(x)))
        ax.set_xticks(ticks)
        ax.set_xticklabels([f"{epoch_times_s[int(t)]:.2f}" for t in ticks])
        ax.set_xlabel("Time (s, epoch centers)")
    else:
        im = ax.imshow(M, origin="lower", aspect="auto", extent=extent, vmin=vmin, vmax=vmax, interpolation="nearest")
        ax.set_xlabel("Epoch index")

    ax.set_ylabel("Amplitude frequency (Hz)")
    if title:
        ax.set_title(title)
    if cbar:
        plt.colorbar(im, ax=ax, label="MI")
    return ax


# ------------------------------
# Minimal example
# ------------------------------

# Synthetic demo (theta phase modulating gamma amplitude)
fs = 1250.0
x = eeg

phase_bands = [(4, 6), (6, 8), (8, 10), (10, 12)]
amp_bands = [(30, 50), (50, 70), (70, 90), (90, 110)]

mi, pf, af, times = compute_pac_comodulogram(x, fs, phase_bands, amp_bands, window_s=1.0, step_s=0.5)

plot_comodulogram(mi, pf, af, agg="mean", title="Comodulogram (mean across epochs)")
plt.show()

plot_time_resolved(mi, pf, af, epoch_times_s=times, phase_idx=2, title=f"Time-resolved MI @ phase~{pf[2]:.1f} Hz")
plt.show()


In [ ]:
"""
Phase–Amplitude Coupling (PAC) comodulogram with Tort Modulation Index (MI)

This module provides:
  1) A robust MI implementation (Tort et al., 2010 style) for limited-time datasets
  2) Time-stepped / event-epoch PAC computation
  3) Publication-ready plotting helpers for comodulograms

Primary entry point: `compute_pac_comodulogram` + `plot_comodulogram`.

Author: ChatGPT
"""
from __future__ import annotations
from typing import Iterable, List, Optional, Sequence, Tuple
import numpy as np
from scipy.signal import butter, filtfilt, hilbert
import matplotlib.pyplot as plt

ArrayLike = np.ndarray

# ------------------------------
# Filtering utilities
# ------------------------------
def _butter_bandpass(low: float, high: float, fs: float, order: int = 4) -> Tuple[np.ndarray, np.ndarray]:
    nyq = 0.5 * fs
    low_n = low / nyq
    high_n = high / nyq
    if low_n <= 0 or high_n >= 1 or low_n >= high_n:
        raise ValueError(f"Invalid band [{low}, {high}] Hz for fs={fs} Hz.")
    b, a = butter(order, [low_n, high_n], btype="bandpass")
    return b, a


def bandpass_filter(x: ArrayLike, fs: float, band: Tuple[float, float], order: int = 4) -> ArrayLike:
    """Zero-phase IIR bandpass using filtfilt.

    Parameters
    ----------
    x : array, shape (n_samples,) or (n_channels, n_samples)
    fs : float
        Sampling rate in Hz.
    band : (low, high)
        Frequency band in Hz.
    order : int
        Butterworth order.
    """
    x = np.asarray(x)
    b, a = _butter_bandpass(band[0], band[1], fs, order)
    if x.ndim == 1:
        return filtfilt(b, a, x)
    elif x.ndim == 2:
        return np.vstack([filtfilt(b, a, xi) for xi in x])
    else:
        raise ValueError("x must be 1D or 2D")


# ------------------------------
# Modulation Index (Tort et al.)
# ------------------------------

def modulation_index_tort(phase_series: ArrayLike, amp_envelope: ArrayLike, n_bins: int = 18, eps: float = 1e-10) -> float:
    """Compute Tort modulation index between a phase time series and an amplitude envelope.

    Parameters
    ----------
    phase_series : array
        Instantaneous phase in radians (e.g., angle(hilbert(theta_band))).
    amp_envelope : array
        Instantaneous amplitude envelope (e.g., abs(hilbert(gamma_band))).
    n_bins : int
        Number of phase bins spanning [-pi, pi).
    eps : float
        Small value to avoid log(0).

    Returns
    -------
    MI : float
        Normalized KL divergence between the phase-binned amplitude distribution and uniform.
    """
    phase = np.asarray(phase_series)
    amp = np.asarray(amp_envelope)
    mask = np.isfinite(phase) & np.isfinite(amp)
    phase = phase[mask]
    amp = amp[mask]
    if phase.size < n_bins * 5:
        return np.nan  # too few samples for a stable estimate

    # Bin by phase
    edges = np.linspace(-np.pi, np.pi, n_bins + 1)
    # Map phase to bin indices [0, n_bins-1]
    idx = np.digitize(phase, edges) - 1
    idx[idx == n_bins] = n_bins - 1

    # Mean amplitude per phase bin
    mean_amp = np.zeros(n_bins)
    counts = np.zeros(n_bins, dtype=int)
    for k in range(n_bins):
        sel = idx == k
        counts[k] = sel.sum()
        if counts[k] > 0:
            mean_amp[k] = amp[sel].mean()
        else:
            mean_amp[k] = 0.0

    if counts.sum() == 0:
        return np.nan

    P = mean_amp / (mean_amp.sum() + eps)
    U = np.full(n_bins, 1.0 / n_bins)

    # KL divergence and normalization
    kl = np.sum(P * (np.log(P + eps) - np.log(U + eps)))
    mi = kl / np.log(n_bins)
    return float(mi)


# ------------------------------
# Epoch helpers
# ------------------------------

def make_sliding_epochs(n_samples: int, fs: float, window_s: float, step_s: Optional[float] = None) -> List[Tuple[int, int]]:
    """Create [start, stop) sample windows covering the signal.

    If step_s is None, defaults to window_s/2.
    """
    if step_s is None:
        step_s = window_s / 2
    w = int(round(window_s * fs))
    s = int(round(step_s * fs))
    if w <= 1:
        raise ValueError("window too small")
    starts = np.arange(0, n_samples - w + 1, s)
    return [(int(a), int(a + w)) for a in starts]


# ------------------------------
# Core PAC computation
# ------------------------------

def compute_pac_comodulogram(
    lfp: ArrayLike,
    fs: float,
    phase_bands: Sequence[Tuple[float, float]],
    amp_bands: Sequence[Tuple[float, float]],
    epochs: Optional[Sequence[Tuple[int, int]]] = None,
    window_s: Optional[float] = None,
    step_s: Optional[float] = None,
    n_bins: int = 18,
    filter_order: int = 4,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, Optional[np.ndarray]]:
    """Compute PAC MI comodulogram over frequency pairs and (optionally) time epochs.

    You can provide explicit `epochs` as a list of (start, stop) sample indices.
    Alternatively, set `window_s` (and optional `step_s`) to tile the signal.

    Parameters
    ----------
    lfp : array, shape (n_samples,)
        LFP / field potential time series.
    fs : float
        Sampling rate in Hz.
    phase_bands : list of (f_lo, f_hi)
        Bands that provide phase (typically low frequencies, e.g., theta 4–12 Hz).
    amp_bands : list of (f_lo, f_hi)
        Bands whose amplitude is modulated (typically higher, e.g., gamma 30–200 Hz).
    epochs : list of (start, stop) in samples, optional
        Event-aligned epochs. If None and window_s given, sliding windows are used.
    window_s, step_s : float, optional
        Sliding window parameters in seconds.
    n_bins : int
        Number of phase bins for MI.
    filter_order : int
        Butterworth order for bandpass.

    Returns
    -------
    mi : array, shape (n_phase, n_amp, n_epochs)
    phase_centers : array, shape (n_phase,)
    amp_centers : array, shape (n_amp,)
    epoch_times_s : array, shape (n_epochs,), optional
        Epoch center times in seconds (None if `epochs` provided explicitly without timing context).
    """
    x = np.asarray(lfp).astype(float)
    if x.ndim != 1:
        raise ValueError("lfp must be 1D")

    # Epochs
    if epochs is None:
        if window_s is None:
            raise ValueError("Provide `epochs` or `window_s`.")
        epochs = make_sliding_epochs(len(x), fs, window_s, step_s)
        epoch_times_s = np.array([(a + b) / 2 / fs for a, b in epochs])
    else:
        epoch_times_s = np.array([(a + b) / 2 / fs for a, b in epochs])

    nP = len(phase_bands)
    nA = len(amp_bands)
    nE = len(epochs)

    # Pre-filter once per band to avoid re-filtering per epoch
    phase_filt = []  # list of instantaneous phase arrays
    for (flo, fhi) in phase_bands:
        xf = bandpass_filter(x, fs, (flo, fhi), order=filter_order)
        phase_filt.append(np.angle(hilbert(xf)))
    phase_filt = np.stack(phase_filt, axis=0)  # (nP, n_samples)

    amp_filt = []  # list of amplitude envelopes per amp band
    for (flo, fhi) in amp_bands:
        xf = bandpass_filter(x, fs, (flo, fhi), order=filter_order)
        amp_filt.append(np.abs(hilbert(xf)))
    amp_filt = np.stack(amp_filt, axis=0)  # (nA, n_samples)

    # Centers for axes labels
    phase_centers = np.array([(flo + fhi) / 2 for (flo, fhi) in phase_bands], dtype=float)
    amp_centers = np.array([(flo + fhi) / 2 for (flo, fhi) in amp_bands], dtype=float)

    # Compute MI per (phase, amp, epoch)
    mi = np.full((nP, nA, nE), np.nan, dtype=float)
    for e, (a, b) in enumerate(epochs):
        # Slightly shrink window to reduce edge artifacts from Hilbert/filter transients
        pad = int(0.05 * (b - a))  # 5% padding removal
        aa = min(a + pad, b)
        bb = max(a, b - pad)
        if bb - aa < 10:
            continue
        for ip in range(nP):
            ph = phase_filt[ip, aa:bb]
            for ia in range(nA):
                am = amp_filt[ia, aa:bb]
                mi[ip, ia, e] = modulation_index_tort(ph, am, n_bins=n_bins)

    return mi, phase_centers, amp_centers, epoch_times_s


# ------------------------------
# Plotting helpers
# ------------------------------

def plot_comodulogram(
    mi: np.ndarray,
    phase_centers: np.ndarray,
    amp_centers: np.ndarray,
    epoch: Optional[int] = None,
    agg: str = "mean",
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    ax: Optional[plt.Axes] = None,
    cbar: bool = True,
    title: Optional[str] = None,
):
    """Plot a standard comodulogram: phase-freq (x) vs amp-freq (y).

    Parameters
    ----------
    mi : array, shape (nP, nA, nE)
    epoch : int or None
        If None, aggregate across epochs using `agg` ("mean" or "max").
    """
    if mi.ndim != 3:
        raise ValueError("mi must be (nP, nA, nE)")

    if epoch is None:
        if agg == "mean":
            M = np.nanmean(mi, axis=2)
        elif agg == "max":
            M = np.nanmax(mi, axis=2)
        else:
            raise ValueError("agg must be 'mean' or 'max'")
    else:
        if not (0 <= epoch < mi.shape[2]):
            raise IndexError("epoch out of range")
        M = mi[:, :, epoch]

    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5), dpi=120)
    im = ax.imshow(
        M.T,
        origin="lower",
        aspect="auto",
        extent=[phase_centers.min(), phase_centers.max(), amp_centers.min(), amp_centers.max()],
        vmin=vmin,
        vmax=vmax,
        interpolation="nearest",
    )
    ax.set_xlabel("Phase frequency (Hz)")
    ax.set_ylabel("Amplitude frequency (Hz)")
    if title:
        ax.set_title(title)
    if cbar:
        plt.colorbar(im, ax=ax, label="Modulation Index (MI)")
    return ax


def plot_time_resolved(
    mi: np.ndarray,
    phase_centers: np.ndarray,
    amp_centers: np.ndarray,
    epoch_times_s: Optional[np.ndarray] = None,
    phase_idx: int = 0,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    ax: Optional[plt.Axes] = None,
    cbar: bool = True,
    title: Optional[str] = None,
):
    """Plot time-resolved PAC for a single phase frequency across amplitude bands.

    Shows amplitude-frequency vs time (epochs on x). Useful for stepping through task time.
    """
    if mi.ndim != 3:
        raise ValueError("mi must be (nP, nA, nE)")
    if not (0 <= phase_idx < mi.shape[0]):
        raise IndexError("phase_idx out of range")

    M = mi[phase_idx, :, :]
    if ax is None:
        fig, ax = plt.subplots(figsize=(7, 4), dpi=120)

    extent = [0, M.shape[1], amp_centers.min(), amp_centers.max()]
    if epoch_times_s is not None and len(epoch_times_s) == M.shape[1]:
        # Use epoch indices but we will relabel ticks with times
        x = np.arange(M.shape[1])
        im = ax.imshow(M, origin="lower", aspect="auto", extent=[x.min(), x.max(), amp_centers.min(), amp_centers.max()], vmin=vmin, vmax=vmax, interpolation="nearest")
        # Relabel x ticks
        ticks = np.linspace(x.min(), x.max(), num=min(8, len(x)))
        ax.set_xticks(ticks)
        ax.set_xticklabels([f"{epoch_times_s[int(t)]:.2f}" for t in ticks])
        ax.set_xlabel("Time (s, epoch centers)")
    else:
        im = ax.imshow(M, origin="lower", aspect="auto", extent=extent, vmin=vmin, vmax=vmax, interpolation="nearest")
        ax.set_xlabel("Epoch index")

    ax.set_ylabel("Amplitude frequency (Hz)")
    if title:
        ax.set_title(title)
    if cbar:
        plt.colorbar(im, ax=ax, label="MI")
    return ax




# ------------------------------
# Convenience helpers to match the paper-style figure
# ------------------------------

def make_bands(fmin: float, fmax: float, width: float, step: float):
    """Create (low, high) bands from fmin..fmax with given width and step.
    Example: make_bands(2, 30, width=2, step=1) -> 2-Hz wide, 1-Hz stepped.
    """
    starts = np.arange(fmin, fmax - width + 1e-9, step)
    bands = [(float(s), float(s + width)) for s in starts]
    return bands


def plot_comodulogram_paper(
    mi: np.ndarray,
    phase_centers: np.ndarray,
    amp_centers: np.ndarray,
    epoch: Optional[int] = None,
    agg: str = "mean",
    y_dash: Optional[float] = 40.0,
    vmin: Optional[float] = None,
    vmax: Optional[float] = None,
    ax: Optional[plt.Axes] = None,
    title: Optional[str] = None,
    cmap: str = "turbo",
    xlim: Optional[Tuple[float, float]] = (0, 100),
    ylim: Optional[Tuple[float, float]] = (0, 100),
):
    """Paper-style comodulogram (blue background, hot spot, dashed 40 Hz line).

    - phase frequency on x (Hz)
    - amplitude frequency on y (Hz)
    - optional dashed line at ~40 Hz
    - defaults to 0..100 Hz on both axes (override with xlim/ylim)
    """
    if mi.ndim != 3:
        raise ValueError("mi must be (nP, nA, nE)")
    if epoch is None:
        M = np.nanmean(mi, axis=2) if agg == "mean" else np.nanmax(mi, axis=2)
    else:
        M = mi[:, :, epoch]

    if ax is None:
        fig, ax = plt.subplots(figsize=(4.0, 4.0), dpi=150)

    im = ax.imshow(
        M.T,
        origin="lower",
        aspect="auto",
        extent=[phase_centers.min(), phase_centers.max(), amp_centers.min(), amp_centers.max()],
        interpolation="nearest",
        vmin=vmin,
        vmax=vmax,
        cmap=cmap,
    )

    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)

    if y_dash is not None:
        ax.axhline(y_dash, linestyle="--", linewidth=1.5, color="white", alpha=0.9)

    ax.set_xlabel("Frequency_phase (Hz)")
    ax.set_ylabel("Frequency_power (Hz)")
    if title:
        ax.set_title(title)

    # Minimal spines/ticks similar to many PAC figures
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.colorbar(im, ax=ax, label="Modulation Index (MI)")
    return ax

fs = 1250.0
x=eeg
phase_bands = make_bands(2, 30, width=2, step=1)
amp_bands   = make_bands(10, 100, width=10, step=2)
mi, pf, af, times = compute_pac_comodulogram(x, fs, phase_bands, amp_bands, window_s=2.0, step_s=1.0)

plot_comodulogram_paper(mi, pf, af, agg="mean", y_dash=40.0, xlim=(0,100), ylim=(0,100), title="PAC comodulogram (paper-style)")
plt.show()


In [ ]:
mwt_zoom